In [5]:
# Cell 1: setup
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
# Cell 2: confirm all 8 day files are in data/raw
from src.data_prep_m2 import check_files, run_m2_split
check_files()

Looking in /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/data/raw
  [seen  ] Monday               Monday-WorkingHours.pcap_ISCX.csv
  [seen  ] Tuesday              Tuesday-WorkingHours.pcap_ISCX.csv
  [seen  ] Wednesday            Wednesday-workingHours.pcap_ISCX.csv
  [unseen] Thu-AM-WebAttacks    Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  [unseen] Thu-PM-Infiltration  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  [unseen] Fri-AM-Bot           Friday-WorkingHours-Morning.pcap_ISCX.csv
  [unseen] Fri-PM-PortScan      Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  [unseen] Fri-PM-DDoS          Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


[]

In [7]:
# Cell 3: build the splits (writes data/processed_m2/)
train, val, test_seen, test_unseen = run_m2_split()

Looking in /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/data/raw
  [seen  ] Monday               Monday-WorkingHours.pcap_ISCX.csv
  [seen  ] Tuesday              Tuesday-WorkingHours.pcap_ISCX.csv
  [seen  ] Wednesday            Wednesday-workingHours.pcap_ISCX.csv
  [unseen] Thu-AM-WebAttacks    Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  [unseen] Thu-PM-Infiltration  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  [unseen] Fri-AM-Bot           Friday-WorkingHours-Morning.pcap_ISCX.csv
  [unseen] Fri-PM-PortScan      Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  [unseen] Fri-PM-DDoS          Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Loading and de-duplicating seen days...
 Monday
  dropped 26935 duplicate rows (529918 -> 502983)
 Tuesday
  dropped 24065 duplicate rows (445909 -> 421844)
 Wednesday
  dropped 81909 duplicate rows (692703 -> 610794)

Loading and de-duplicating unseen days...
 Thu-AM-WebAttacks
  dropped 6066 dupl

In [8]:
# Cell 4: multi-seed validation report (train/val only, nothing saved)
from src import m2_experiments as m2
allres, summ = m2.run_val()


########## seed 42 ##########
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

=== validation, seed 42 ===

AUROC
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

RECALL by attack family
                      n  Autoencoder @0.1%  Autoencoder @1.0%  RF-IDS @0.1%  RF-IDS @1.0%
item                                                                                     
ALL attacks       30432             0.8391             0.8946        0.9940        0.9995
DoS GoldenEye      1542             0.2575             0.4754        0.9604        0.9948
DoS Hulk          25927             0.9544             0.9735        0.9991        0.9997
DoS Slowhttptest    784             0.0612             0.9120        0.8763        0.9987
Do

In [11]:
from src import diagnostics as dg
_ = dg.port_only_check("data/processed_m2")   # is FTP/SSH solvable by port alone?
_ = dg.error_analysis("data/processed_m2")    # why does the AE miss brute force?
_ = dg.rf_leakage_check("data/processed_m2")  # RF still ~1.0 with top features removed?

=== Destination Port ONLY (RF, 50 trees) - AUROC 0.9842 ===
  distinct score values on val: 5
  target FPR  achieved FPR   recall  benign tied @thr
          1%        0.0011   0.0292            0.0299
          5%        0.0311   1.0000            0.9689
         10%        0.0311   1.0000            0.9689
         20%        0.0311   1.0000            0.9689

  Top destination ports by share of each class (val):
    benign: 53=61.2%, 443=6.2%, 80=3.0%, 88=0.5%, 389=0.4%
    attack: 80=95.5%, 21=2.9%, 22=1.6%, 444=0.0%
  If benign port 80 > 1% and attack port 80 is high, the 1% threshold sits
  on port 80's score, strict `>` flags nothing, and 0% recall is a tie artifact.
--- Autoencoder ---
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

AE threshold 0.0449, achieved FPR 0.0100
att

In [1]:
from src import diagnostics as dg
abl = dg.context_ablation("data/processed_m2", seeds=(42,))


--- RF-IDS | without port (current) | seed 42 ---

--- Autoencoder | without port (current) | seed 42 ---
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

--- RF-IDS | with port | seed 42 ---

--- Autoencoder | with port | seed 42 ---
  epoch   1/30  train MSE: 0.186104
  epoch   5/30  train MSE: 0.053016
  epoch  10/30  train MSE: 0.041032
  epoch  15/30  train MSE: 0.031367
  epoch  20/30  train MSE: 0.023586
  epoch  25/30  train MSE: 0.017405
  epoch  30/30  train MSE: 0.022291

=== Context ablation (val) ===
      model  seed                variant  auroc    thr  achieved_fpr  recall@1%
     RF-IDS    42 without port (current) 0.9998 0.0150        0.0082     0.9995
Autoencoder    42 without port (current) 0.9888 0.0449        0.0100     0.8946
     RF-IDS    42              with 

In [2]:
from src import m2_experiments as m2
dets2, thr2 = m2.load_official()
val = m2._load("val")
y = val["y"]
for name, d in dets2.items():
    flag = d["score"](val["X"]) > thr2[name][0.01]
    tp = int((flag & (y == 1)).sum()); fp = int((flag & (y == 0)).sum())
    fn = int((~flag & (y == 1)).sum()); tn = int((~flag & (y == 0)).sum())
    n = tp + fp + fn + tn
    print(name, "acc %.3f prec %.3f rec %.4f f1 %.3f fpr %.4f fnr %.4f" % (
        (tp+tn)/n, tp/(tp+fp), tp/(tp+fn), 2*tp/(2*tp+fp+fn), fp/(fp+tn), fn/(fn+tp)))

Autoencoder acc 0.977 prec 0.932 rec 0.8946 f1 0.913 fpr 0.0100 fnr 0.1054
RF-IDS acc 0.993 prec 0.949 rec 0.9995 f1 0.973 fpr 0.0082 fnr 0.0005


In [3]:
import time, numpy as np
from pathlib import Path
from src import m2_experiments as m2
from src.autoencoder import fit_scaler, train_autoencoder
from src.rf_ids import train_rf

train, val = m2._load("train"), m2._load("val")
Xb = train["X"][train["y"] == 0]

# training time, 3 runs each (seed 42)
ae_t, rf_t = [], []
for _ in range(3):
    t = time.perf_counter(); sc = fit_scaler(Xb); train_autoencoder(sc.transform(Xb), seed=42); ae_t.append(time.perf_counter() - t)
    t = time.perf_counter(); train_rf(train["X"], train["y"], seed=42); rf_t.append(time.perf_counter() - t)
print("training s  AE %.1f +/- %.1f | RF %.1f +/- %.1f" % (np.mean(ae_t), np.std(ae_t), np.mean(rf_t), np.std(rf_t)))

# inference per 1,000 flows on the official models: 1 warm-up, then 5 timed passes over validation
dets, _ = m2.load_official()
X = val["X"]; n = len(X)
for name, d in dets.items():
    d["score"](X[:5000])
    ts = []
    for _ in range(5):
        t = time.perf_counter(); d["score"](X); ts.append(time.perf_counter() - t)
    per1k = np.array(ts) / (n / 1000) * 1000   # ms per 1,000 flows
    print("%s inference %.3f +/- %.3f ms per 1,000 flows (n=%d)" % (name, per1k.mean(), per1k.std(), n))

# saved size
for f in ["autoencoder.pt", "scaler.joblib", "rf.joblib"]:
    print(f, "%.1f KB" % (Path("models_m2", f).stat().st_size / 1024))

  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
training s  AE 106.0 +/- 4.9 | RF 76.9 +/- 0.4
Autoencoder inference 0.373 +/- 0.164 ms per 1,000 flows (n=230222)
RF-IDS inference 1.116 +/- 0.008 ms per 1,000 flows (n=230222)
autoencoder.pt 66.6 KB
scaler.joblib 2.4 KB
rf.joblib 43364.6 KB


In [12]:
# Cell 5: ALREADY RUN, DO NOT RERUN. Official models + thresholds
# dets, thr = m2.save_official()

  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

=== validation, official seed 42 ===

AUROC
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

RECALL by attack family
                      n  Autoencoder @0.1%  Autoencoder @1.0%  RF-IDS @0.1%  RF-IDS @1.0%
item                                                                                     
ALL attacks       30432             0.8391             0.8946        0.9940        0.9995
DoS GoldenEye      1542             0.2575             0.4754        0.9604        0.9948
DoS Hulk          25927             0.9544             0.9735        0.9991        0.9997
DoS Slowhttptest    784             0.0612             0.9120        0.8763        0.9987
DoS slowloris       807 

In [13]:
print(thr)

{'Autoencoder': {0.01: 0.04489630088210106, 0.001: 0.26539134979248047}, 'RF-IDS': {0.01: 0.015, 0.001: 0.175}}


In [14]:
import importlib
from src import m2_experiments as m2
importlib.reload(m2)
print("new version loaded:", hasattr(m2, "reanalyze_saved"))

new version loaded: True


In [15]:
dets2, thr2 = m2.load_official()
print(thr2)   # should equal the thresholds you pasted
val = m2._load("val")
m2.show(m2.evaluate_split(dets2, thr2, val), "validation via load_official (round-trip check)")

{'Autoencoder': {0.01: 0.04489630088210106, 0.001: 0.26539134979248047}, 'RF-IDS': {0.01: 0.015, 0.001: 0.175}}

=== validation via load_official (round-trip check) ===

AUROC, all flows
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

AUROC by day (that day's attacks vs that day's benign)
detector       n  Autoencoder  RF-IDS
item                                 
Tuesday    63249       0.8946  0.9999
Wednesday  91589       0.9934  0.9997

AUROC by attack family (family vs benign flows from the same day)
detector              n  Autoencoder  RF-IDS
item                                        
DoS GoldenEye      1542       0.9696  0.9979
DoS Hulk          25927       0.9954  0.9998
DoS Slowhttptest    784       0.9916  0.9994
DoS slowloris       807       0.9768  0.9999
FTP-Patator         889       0.9016  0.9999
Heartbleed            1       0.9997  0.9994
SSH-Patator         482       0.8815  0.9999

RECALL by attack f

In [5]:
# Cell 6: Scores the test sets, once
# Ran on Saturday, November 21. Do Not run again


# m2.score_test()


=== TEST, seen days (Mon-Wed held-out slice) ===

AUROC, all flows
detector       n  Autoencoder  RF-IDS
item                                 
overall   230259       0.8456  0.9996

AUROC by day (that day's attacks vs that day's benign)
detector       n  Autoencoder  RF-IDS
item                                 
Tuesday    63262       0.8947  0.9979
Wednesday  91587       0.8472  0.9996

AUROC by attack family (family vs benign flows from the same day)
detector              n  Autoencoder  RF-IDS
item                                        
DoS GoldenEye      1544       0.9826  0.9999
DoS Hulk          25928       0.8306  0.9996
DoS Slowhttptest    785       0.9940  0.9991
DoS slowloris       809       0.9792  0.9996
FTP-Patator         890       0.9001  0.9970
Heartbleed            3       0.9997  0.9996
SSH-Patator         484       0.8848  0.9997

RECALL by attack family at the FROZEN threshold (headline)
                      n  Autoencoder @0.1%  Autoencoder @1.0%  RF-IDS @0.1%  R

(       detector  target_fpr            scope           item       n     value
 0   Autoencoder         NaN            auroc        overall  230259  0.845612
 1   Autoencoder         NaN        auroc_day        Tuesday   63262  0.894700
 2   Autoencoder         NaN        auroc_day      Wednesday   91587  0.847245
 3   Autoencoder         NaN     auroc_family  DoS GoldenEye    1544  0.982565
 4   Autoencoder       0.010  recall_rethresh  DoS GoldenEye    1544  0.637306
 ..          ...         ...              ...            ...     ...       ...
 91       RF-IDS       0.001           recall     Heartbleed       3  1.000000
 92       RF-IDS       0.001           recall    SSH-Patator     484  0.971074
 93       RF-IDS       0.001              fpr         Monday   75410  0.000822
 94       RF-IDS       0.001              fpr        Tuesday   61888  0.001406
 95       RF-IDS       0.001              fpr      Wednesday   62518  0.000928
 
 [96 rows x 6 columns],
         detector  target_

In [1]:
# Cell 7 (post-hoc, exploratory): does Hulk recall fall late in the capture?
# Uses saved test scores; scores train/val Hulk rows with the frozen models.
# Rebuilds capture order from the raw Wednesday file (the split CSVs are shuffled).
from pathlib import Path
import numpy as np, pandas as pd
from src import m2_experiments as m2
from src.data_prep import dedupe
from src.data_prep_m2 import _find_file, _load_day, SEEN_FILES
from src.evaluate import NON_FEATURE_COLS

def fp(X):  # row fingerprint, float32 so CSV round-trip noise cannot change it
    return pd.util.hash_pandas_object(pd.DataFrame(np.asarray(X, dtype=np.float32)), index=False).to_numpy()

raw = dedupe(_load_day(_find_file(Path("data/raw"), SEEN_FILES["Wednesday"]), "Wednesday"))
h = raw[raw["Label"] == "DoS Hulk"].reset_index(drop=True)
h_pos = (np.arange(len(h)) + 0.5) / len(h)                 # 0 = start of Hulk run, 1 = end
cols = [c for c in raw.columns if c not in NON_FEATURE_COLS]
Xh = h[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
ok = Xh.notna().all(axis=1).to_numpy()
pos_of = pd.Series(h_pos[ok], index=fp(Xh[ok].to_numpy(float)))
pos_of = pos_of[~pos_of.index.duplicated()]

dets, thr = m2.load_official()
t_ae, t_rf = thr["Autoencoder"][0.01], thr["RF-IDS"][0.01]
saved = m2.load_test_scores()["seen"]

parts = []
for split in ("train", "val", "test_seen"):
    d = m2._load(split)
    m = d["label"] == "DoS Hulk"
    X = d["X"][m]
    if split == "test_seen":
        assert (saved["idx"].to_numpy() == d["idx"]).all(), "saved scores not aligned with test_seen"
        ae, rf = saved.loc[m, "score_autoencoder"].to_numpy(), saved.loc[m, "score_rf_ids"].to_numpy()
    else:
        ae, rf = dets["Autoencoder"]["score"](X), dets["RF-IDS"]["score"](X)
    parts.append(pd.DataFrame({"split": split, "pos": pos_of.reindex(fp(X)).to_numpy(),
                               "ae": ae > t_ae, "rf": rf > t_rf, "ae_ratio": ae / t_ae}))
t = pd.concat(parts, ignore_index=True)
print("Hulk rows matched to capture position:", f"{t['pos'].notna().mean():.1%}  (should be ~100%)")
t = t.dropna(subset=["pos"])
t["bin"] = (np.floor(t["pos"] * 20) / 20).round(2)          # 20 bins of 5% of the Hulk run
out = t.groupby("bin").agg(split=("split", lambda s: s.mode()[0]), n=("ae", "size"),
                           AE_recall=("ae", "mean"), RF_recall=("rf", "mean"),
                           AE_median_score_over_thr=("ae_ratio", "median"))
print("\nDoS Hulk recall at the frozen 1% thresholds, by position in the capture")
print("(RF rows in 'train' are in-sample; the AE never trains on attacks)")
print(out.to_string(float_format=lambda v: f"{v:.3f}"))

  dropped 81909 duplicate rows (692703 -> 610794)
Hulk rows matched to capture position: 100.0%  (should be ~100%)

DoS Hulk recall at the frozen 1% thresholds, by position in the capture
(RF rows in 'train' are in-sample; the AE never trains on attacks)
          split     n  AE_recall  RF_recall  AE_median_score_over_thr
bin                                                                  
0.00      train  8640      0.765      1.000                     6.420
0.05      train  8643      0.790      1.000                     6.501
0.10      train  8642      0.822      1.000                     7.197
0.15      train  8643      0.933      1.000                     8.041
0.20      train  8642      0.959      1.000                     8.520
0.25      train  8642      0.965      1.000                     8.304
0.30      train  8642      0.962      1.000                     8.544
0.35      train  8643      0.953      1.000                     8.599
0.40      train  8642      0.962      1.000  

In [2]:
# Cell 8 (post-hoc, exploratory): why is benign FPR so high on Fri-PM-DDoS?
# Uses saved test scores and the frozen thresholds only. Nothing is re-scored.
import numpy as np, pandas as pd
from src import m2_experiments as m2
from src.evaluate import load_split, NON_FEATURE_COLS

thr = m2._load_thr(m2.MODELS_DIR)
sc = m2.load_test_scores()["unseen"]
fc = None
def numeric(df):
    return df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

un = load_split(m2.DATA_DIR, "test_unseen")
fc = [c for c in un.columns if c not in NON_FEATURE_COLS]
F = numeric(un.iloc[sc["idx"].to_numpy()][fc]).reset_index(drop=True)
ae = (sc["score_autoencoder"] > thr["Autoencoder"][0.01]).to_numpy()
rf = (sc["score_rf_ids"] > thr["RF-IDS"][0.01]).to_numpy()
ben = (sc["y"] == 0).to_numpy()
day = sc["day"].to_numpy()

rows = []
for d in sorted(set(day)):
    m = ben & (day == d)
    rows.append({"day": d, "benign n": m.sum(), "AE FPR": ae[m].mean(), "RF FPR": rf[m].mean(),
                 "both flag": (ae & rf)[m].mean(), "either flags": (ae | rf)[m].mean()})
print("Benign flows flagged at the frozen 1% thresholds, by day")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

D = "Fri-PM-DDoS"
m = ben & (day == D)
tr = load_split(m2.DATA_DIR, "train")
trb = numeric(tr.loc[tr["binary_label"] == "BENIGN", fc])
mu, sd = trb.median(), trb.std().replace(0, np.nan)

def gaps(a, b, label, k=8):
    g = ((a.median() - b.median()) / sd).dropna()
    print(f"\n{label}: largest median gaps (train-benign sd units)")
    print(g.reindex(g.abs().sort_values(ascending=False).index)[:k].round(2).to_string())

gaps(F[m], trb, f"{D} benign vs Mon-Wed train benign")
gaps(F[m & ae], F[m & ~ae], f"{D} benign, AE-flagged vs not flagged")
gaps(F[m & rf], F[m & ~rf], f"{D} benign, RF-flagged vs not flagged")

Benign flows flagged at the frozen 1% thresholds, by day
                day  benign n  AE FPR  RF FPR  both flag  either flags
         Fri-AM-Bot    182096  0.0111  0.0060     0.0007        0.0163
        Fri-PM-DDoS     95068  0.2465  0.1733     0.1531        0.2667
    Fri-PM-PortScan    123083  0.0101  0.0068     0.0009        0.0160
  Thu-AM-WebAttacks    162036  0.0088  0.0064     0.0007        0.0146
Thu-PM-Infiltration    252754  0.0350  0.0418     0.0159        0.0609

Fri-PM-DDoS benign vs Mon-Wed train benign: largest median gaps (train-benign sd units)
Bwd Packet Length Mean   -0.25
Avg Bwd Segment Size     -0.25
Min Packet Length         0.22
Fwd Packet Length Min     0.18
Bwd Packet Length Max    -0.09
Bwd Packet Length Min     0.09
Average Packet Size      -0.05
Packet Length Mean       -0.03

Fri-PM-DDoS benign, AE-flagged vs not flagged: largest median gaps (train-benign sd units)
Fwd Packet Length Mean    10.58
Avg Fwd Segment Size      10.58
Fwd Packet Length Std   

In [3]:
# Cell 9 (post-hoc, exploratory): recall with and without rows copied from train.
# Same overlap rule as data_prep_m2 (exact feature vector, Destination Port excluded).
import numpy as np, pandas as pd
from src import m2_experiments as m2
from src.evaluate import load_split

thr = m2._load_thr(m2.MODELS_DIR)
sc = m2.load_test_scores()["unseen"]
tr, un = load_split(m2.DATA_DIR, "train"), load_split(m2.DATA_DIR, "test_unseen")
fc = [c for c in tr.columns if c not in ("Label", "binary_label", "day", "Destination Port")]
h_tr = set(pd.util.hash_pandas_object(tr[fc], index=False))
ov_all = pd.util.hash_pandas_object(un[fc], index=False).isin(h_tr).to_numpy()
print("Check, overlap by day (should match the Cell 3 printout: 8.44 / 6.87 / 8.20 / 5.26 / 2.50 %)")
print((pd.Series(ov_all).groupby(un["day"].to_numpy()).mean() * 100).round(2).to_string())

ov = ov_all[sc["idx"].to_numpy()]
y, lab, day = sc["y"].to_numpy(), sc["label"].to_numpy(), sc["day"].to_numpy()
flags = {"AE": (sc["score_autoencoder"] > thr["Autoencoder"][0.01]).to_numpy(),
         "RF": (sc["score_rf_ids"] > thr["RF-IDS"][0.01]).to_numpy()}

rows = []
for L in sorted(set(lab[y == 1])):
    m = lab == L
    r = {"family": L, "n": m.sum(), "n in train": (m & ov).sum()}
    for k, f in flags.items():
        r[f"{k} all"] = f[m].mean()
        r[f"{k} no-overlap"] = f[m & ~ov].mean() if (m & ~ov).any() else np.nan
    rows.append(r)
m = y == 1
r = {"family": "ALL attacks", "n": m.sum(), "n in train": (m & ov).sum()}
for k, f in flags.items():
    r[f"{k} all"], r[f"{k} no-overlap"] = f[m].mean(), f[m & ~ov].mean()
rows.append(r)
print("\nRecall at the frozen 1% thresholds, all rows vs rows NOT found in train")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

rows = []
for d in sorted(set(day)):
    m = (y == 0) & (day == d)
    rows.append({"day": d, "benign n": m.sum(), "n in train": (m & ov).sum(),
                 "AE FPR all": flags["AE"][m].mean(), "AE no-overlap": flags["AE"][m & ~ov].mean(),
                 "RF FPR all": flags["RF"][m].mean(), "RF no-overlap": flags["RF"][m & ~ov].mean()})
print("\nBenign FPR, all rows vs rows NOT found in train")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

Check, overlap by day (should match the Cell 3 printout: 8.44 / 6.87 / 8.20 / 5.26 / 2.50 %)
Fri-AM-Bot             8.20
Fri-PM-DDoS            2.50
Fri-PM-PortScan        5.26
Thu-AM-WebAttacks      8.44
Thu-PM-Infiltration    6.87

Recall at the frozen 1% thresholds, all rows vs rows NOT found in train
                    family      n  n in train  AE all  AE no-overlap  RF all  RF no-overlap
                       Bot   1948           9  0.0226         0.0227  0.0000         0.0000
                      DDoS 128014           0  0.6275         0.6275  0.6490         0.6490
              Infiltration     36           0  0.6111         0.6111  0.6944         0.6944
                  PortScan  90694           0  0.1570         0.1570  0.0163         0.0163
  Web Attack - Brute Force   1470           0  0.0014         0.0014  0.8646         0.8646
Web Attack - Sql Injection     21           0  0.0000         0.0000  0.8095         0.8095
          Web Attack - XSS    652           0  0.0